# Huấn luyện YOLO nhận diện mệnh giá tiền Việt Nam

Trước khi chạy: **Runtime → Change runtime type → T4 GPU → Save**, sau đó **Runtime → Run all**. Khi được hỏi, chọn tệp `dataset_yolo.zip`. Khi xong, trình duyệt tự tải về `ket_qua_yolo_colab.zip`.

## 1. Kiểm tra GPU và cài thư viện

In [ ]:
import torch
assert torch.cuda.is_available(), 'CHƯA BẬT GPU: vào Runtime > Change runtime type > T4 GPU, rồi chạy lại.'
print('GPU:', torch.cuda.get_device_name(0))
!pip install -q ultralytics==8.4.202

## 2. Tải lên và giải nén dữ liệu

In [ ]:
import os, zipfile
from google.colab import files
if not os.path.exists('/content/dataset_yolo.zip'):
    files.upload()   # chọn tệp dataset_yolo.zip
with zipfile.ZipFile('/content/dataset_yolo.zip') as z:
    z.extractall('/content')
so_anh = {s: len(os.listdir(f'/content/dataset_yolo/{s}/images')) for s in ('train', 'valid', 'test')}
print(so_anh)
assert so_anh == {'train': 559, 'valid': 439, 'test': 219}, 'Dữ liệu không đầy đủ, hãy tải lại tệp zip.'

## 3. Huấn luyện (khoảng 30-60 phút)

Giữ tab này mở trong lúc chạy. Tập test **không** được dùng ở đây; tập test sẽ được đánh giá sau, trên máy tính của nhóm.

In [ ]:
import json, time, torch, ultralytics
from ultralytics import YOLO
bat_dau = time.time()
model = YOLO('yolo11n.pt')
model.train(data='/content/dataset_yolo/tien.yaml', epochs=60, patience=15, batch=16, imgsz=640, device=0,
            seed=0, deterministic=True, workers=2, cache='ram',
            project='/content/ket_qua', name='yolo11n', exist_ok=True, plots=True)
thong_tin = {'gpu': torch.cuda.get_device_name(0), 'ultralytics': ultralytics.__version__, 'torch': torch.__version__,
             'thoi_gian_huan_luyen_phut': round((time.time() - bat_dau) / 60, 1)}
json.dump(thong_tin, open('/content/ket_qua/yolo11n/thong_tin.json', 'w'), indent=2)
print(thong_tin)

## 4. Tải kết quả về máy

In [ ]:
import shutil
from google.colab import files
assert os.path.exists('/content/ket_qua/yolo11n/weights/best.pt'), 'Chưa có best.pt: bước huấn luyện chưa xong.'
shutil.make_archive('/content/ket_qua_yolo_colab', 'zip', '/content/ket_qua')
files.download('/content/ket_qua_yolo_colab.zip')